# UK Exports Predictor

This notebook is the project write-up. It learns from global HS6 goods trade what Flow to expect between any Exporter, Importer and Product, applies that to the UK to estimate **Export Potential** for every Product–Market pair, and ranks positive **Export Gaps** as **Opportunities**.

All reusable logic lives in the `exportspredictor` package, so the notebook and the API compute features identically. The notebook only calls it.

## Parameters

`FIXTURE_MODE = True` runs everything on a tiny synthetic world in seconds; the test suite executes the notebook this way. Real-data mode arrives with the BACI loading work.

In [ ]:
FIXTURE_MODE = True

In [ ]:
import polars as pl

from exportspredictor import artefacts
from exportspredictor.fixture import build_world
from exportspredictor.paths import artefacts_dir
from exportspredictor.pipeline import build_artefacts

## 1. Tracer bullet: the fixture world

Before any real data, we prove that every layer connects: data, features, fold models, artefacts, API and app. The fixture world has 4 Exporters (including the UK), 3 Markets, 5 Products and 4 years. Flows follow a gravity-like rule, and the UK–Japan Trade Agreement switches on in 2022, so the world contains a policy change for the model to learn from.

In [ ]:
if not FIXTURE_MODE:
    raise NotImplementedError("Real-data mode is not available yet; set FIXTURE_MODE = True.")
world = build_world()
world.flows.head()

In [ ]:
world.policy.filter(pl.col("exporter") == "GBR").pivot(on="year", index="importer", values="trade_agreement")

## 2. Training and out-of-fold scoring

The model is gradient boosting (XGBoost) with a Tweedie objective, which suits a non-negative target with many zeros. This extends the course's tree-ensemble material. Three guardrails are already in place:

- **Time split** (course: train/validation/test splits and leakage): fold models fit on years before the Baseline year and score the Baseline year.
- **Out-of-fold UK scoring** (ADR 0001, beyond the course's k-fold cross-validation): each UK Product–Market pair gets its Export Potential from the fold model that never saw it, so the Export Gap is not shrunk by memorisation.
- **Monotone constraint** on the Trade Agreement feature (ADR 0002): adding a Trade Agreement never lowers Export Potential. The model *associates* an agreement with higher Flows; it does not show cause.

In [ ]:
fixture = build_artefacts(world, source="fixture")
fixture.metadata

The Baseline table compares Export Potential with Actual Exports for every UK pair; a positive Export Gap is an Opportunity.

In [ ]:
fixture.baseline.sort("export_gap_usd", descending=True)

## 3. Saving the artefacts

The API serves exactly what was evaluated here: a versioned artefact folder with the fold models, the Baseline table, lookups and a metadata file.

In [ ]:
saved_to = artefacts.save(fixture, artefacts_dir())
saved_to